# Solar filament instance segmentation — final reproduction
Shengkai Zhu, independent researcher. This notebook reconstructs the five supplied checkpoints and runs real inference for the two accepted ens5c submissions. It does not train a model. The reference CSVs are used only to compare predictions after inference.

Use Python 3.12 and a CUDA GPU. Install `requirements.txt` in a fresh environment; supply the official competition dataset after accepting its terms. No test annotations are required.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = Path.cwd().resolve()
assert (ROOT / 'weights/manifest.json').is_file(), 'Run from repository root'
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=ROOT, check=True)
run('scripts/restore_weights.py')

## Official data
Set `FILAMENT_DATA_ROOT` to the folder containing `train/` and `test/`. On Kaggle the competition mount is detected automatically. The image stays grayscale; input size is 2048 × 2048.

In [ ]:
expected = ROOT / 'data/MAGFiLO_1.0_Kaggle_2026'
if not expected.exists():
    supplied = os.environ.get('FILAMENT_DATA_ROOT')
    if supplied:
        source = Path(supplied).resolve()
    else:
        matches = list(Path('/kaggle/input').rglob('MAGFiLO_1.0_Annotations_kaggle2026_train.json'))
        assert len(matches) == 1, 'Set FILAMENT_DATA_ROOT to the official dataset folder'
        source = matches[0].parent.parent
    assert (source / 'test/test_images').is_dir()
    expected.parent.mkdir(exist_ok=True)
    expected.symlink_to(source, target_is_directory=True)
run('scripts/prepare_cache.py', '--workers', 4, '--spawn')
original = json.loads((ROOT / 'metadata/original_split.json').read_text())
generated = json.loads((ROOT / 'data/cache/index.json').read_text())
assert all(generated[key] == original[key] for key in ['train', 'val', 'test'])
print('Original observation split verified:', len(original['train']), len(original['val']))

## Fresh predictions
Four-flip TTA, equal-weight five-model probability average, contact-aware instance separation, and the supplied gradient-boosting rescorer. Both rescorers share a single model inference pass. No reference CSV is used as input.

In [ ]:
run('scripts/reproduce_predictions.py', '--device', 'cuda')

## Verify both CSVs
The audit checks 180 image IDs, unique instance IDs, valid nonempty COCO RLE, and no overlapping instances. The reproduction report records byte hashes and decoded mask agreement. Floating-point differences between GPU/software stacks can affect boundary pixels.

In [ ]:
for variant in ['match', 'eiou']:
    run('scripts/final_submission_audit.py', '--csv', f'outputs/submission_ens5c_rescore_{variant}.csv',
        '--out', f'outputs/{variant}_audit.json')
print((ROOT / 'outputs/reproduction_report.json').read_text())

## Evaluation and training evidence
The technical report explains the observation split, annotation disagreement and limits of the nested rescorer validation. `runs/*/config.json` and `log.csv` record each training run. The stored evaluation report preserve local evidence. The validation score is not an independent estimate after model selection. Research models not present in the selected submission are excluded from this reproduction.